# LSTM Stock Market Forecasting

This notebook builds a daily closing-price forecasting workflow around a Long
Short-Term Memory (LSTM) network. Technical indicators provide the input
signals. Principal Component Analysis (PCA) can be enabled as an optional
preprocessing step through one configuration value.

The goal is a clear, reproducible experiment rather than a trading strategy.

## Workflow

```text
market data -> technical indicators -> chronological split
            -> training-only scaling -> optional PCA -> sequences -> LSTM
```

The validation period controls early stopping. The test period is used only
after training has finished.

In [ ]:
from pathlib import Path
import os
import random

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
import yfinance as yf
import ta

from sklearn.decomposition import PCA
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error, mean_squared_error
from sklearn.preprocessing import StandardScaler
from tensorflow import keras


TICKER = "AAPL"
START_DATE = "2015-01-01"
END_DATE = "2023-01-01"
SEQUENCE_LENGTH = 60

# Keep this as None for the standard LSTM input pipeline.
# Set an integer, for example 4, to enable PCA with that many components.
PCA_COMPONENTS = None

TRAIN_FRACTION = 0.70
VALIDATION_FRACTION = 0.15
EPOCHS = 100
BATCH_SIZE = 64
SEED = 42


def set_seed(seed):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    tf.keras.utils.set_random_seed(seed)


set_seed(SEED)

## 1. Market data

The notebook downloads daily OHLCV data with `yfinance` and keeps a local CSV
cache. The cache makes later runs independent of small upstream API changes.

In [ ]:
def load_market_data(ticker, start_date, end_date, cache_dir="data"):
    cache_dir = Path(cache_dir)
    cache_path = cache_dir / f"{ticker.lower()}_{start_date}_{end_date}.csv"

    if cache_path.exists():
        frame = pd.read_csv(cache_path, index_col="Date", parse_dates=["Date"])
    else:
        frame = yf.download(
            ticker,
            start=start_date,
            end=end_date,
            auto_adjust=False,
            progress=False,
        )
        if isinstance(frame.columns, pd.MultiIndex):
            frame = frame.xs(ticker, axis=1, level=-1)
        frame.index.name = "Date"
        cache_dir.mkdir(parents=True, exist_ok=True)
        frame.to_csv(cache_path)

    required = ["Open", "High", "Low", "Close", "Volume"]
    missing = set(required) - set(frame.columns)
    if missing:
        raise ValueError(f"Missing market-data columns: {sorted(missing)}")
    return frame[required].sort_index().dropna().copy()


market_data = load_market_data(TICKER, START_DATE, END_DATE)
market_data.tail()

## 2. Technical indicators

The feature set combines price, trend, momentum, volatility, and trend-strength
signals. Rolling indicators introduce missing rows at the beginning, which are
removed after all features have been calculated.

In [ ]:
FEATURE_COLUMNS = [
    "Close",
    "MA50",
    "SMA5",
    "RSI",
    "MACD",
    "BollingerUpper",
    "BollingerLower",
    "ATR",
    "ADX",
    "STD10",
]


def add_technical_indicators(frame):
    result = frame.copy()
    close = result["Close"]
    high = result["High"]
    low = result["Low"]

    result["MA50"] = close.rolling(50).mean()
    result["SMA5"] = close.rolling(5).mean()
    result["STD10"] = close.rolling(10).std()
    result["MACD"] = (
        close.ewm(span=12, adjust=False).mean()
        - close.ewm(span=26, adjust=False).mean()
    )

    ma10 = close.rolling(10).mean()
    result["BollingerUpper"] = ma10 + 2 * result["STD10"]
    result["BollingerLower"] = ma10 - 2 * result["STD10"]

    change = close.diff()
    average_gain = change.clip(lower=0).rolling(14).mean()
    average_loss = -change.clip(upper=0).rolling(14).mean()
    relative_strength = average_gain / average_loss.replace(0, np.nan)
    result["RSI"] = 100 - 100 / (1 + relative_strength)

    true_range = pd.concat(
        [
            high - low,
            (high - close.shift()).abs(),
            (low - close.shift()).abs(),
        ],
        axis=1,
    ).max(axis=1)
    result["ATR"] = true_range.rolling(14).mean()
    result["ADX"] = ta.trend.ADXIndicator(
        high=high,
        low=low,
        close=close,
        window=14,
    ).adx()

    return result.dropna().copy()


model_data = add_technical_indicators(market_data)
print(f"Rows available for modelling: {len(model_data):,}")
model_data[FEATURE_COLUMNS].tail()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(model_data.index, model_data["Close"], label="Close", linewidth=1.3)
ax.plot(model_data.index, model_data["MA50"], label="50-day moving average", linewidth=1)
ax.set(title=f"{TICKER} closing price", xlabel="Date", ylabel="Price")
ax.legend()
ax.grid(alpha=0.25)
plt.tight_layout()